In [20]:
import swisseph as swe
import math



def get_julian(year, month, day, hour, minute):
    return swe.julday(year, month, day, hour + minute/60.0)

hrm =  4 + (15/60)

jd = get_julian(2025,12,14,12,51)  
#print(jd)


def calculate_ascendant_swiss(jd_ut, lat, lon):
    """
    Most accurate method using Swiss Ephemeris houses
    """
    try:
        # Use Placidus house system (most common in Vedic astrology)
        houses = swe.houses(jd_ut, lat, lon, b'P')  # b'P' = Placidus
        
        # houses[0] contains house cusps, index 0 is Ascendant (1st house)
        ascendant = houses[0][0]
        ascendant_sidereal = (ascendant - ayanamsa) % 360

        
        return ascendant_sidereal
        
    except Exception as e:
        print(f"Swiss Ephemeris house calculation failed: {e}")
print(calculate_ascendant_swiss(jd, lat=19.0550,lon=72.8777))



# Calculate positions at midpoint with Lahiri ayanamsa
ayanamsa = swe.get_ayanamsa(jd)
sun_pos, _ = swe.calc(jd, swe.SUN)
sun_pos_, _ = swe.calc_ut(jd, swe.SUN)
moon_pos, _ = swe.calc(jd, swe.MOON)
saturn_pos,_ = swe.calc(jd, swe.SATURN)
jupiter_pos,_ = swe.calc(jd, swe.JUPITER)
mercury_pos,_ = swe.calc(jd, swe.MERCURY)
mars_pos,_ = swe.calc(jd,swe.MARS)
venus_pos,_ = swe.calc(jd, swe.VENUS)
rahu_pos,_ = swe.calc(jd, swe.TRUE_NODE)
ketu_pos = rahu_pos[0]
print(f"Sun:{(sun_pos[0]-ayanamsa) % 360}")
print(f"Sun with UTC calculation : {(sun_pos_[0]-ayanamsa) % 360}")
print(f"Moon:{(moon_pos[0]-ayanamsa) % 360}")
print(f"Saturn: {(saturn_pos[0]-ayanamsa) % 360}")
print(f"jupiter : {(jupiter_pos[0]-ayanamsa) % 360}")
print(f"mercury : {(mercury_pos[0]-ayanamsa) % 360}")
print(f"venus : {(venus_pos[0]-ayanamsa) % 360}")
print(f"rahu : {(rahu_pos[0]-ayanamsa) % 360}")
print(f"ketu : {(ketu_pos)}")

ayanamsa = swe.get_ayanamsa_ut(jd)
print(swe.calc(jd, swe.JUPITER))
print(swe.calc(jd, swe.SUN))
print(swe.calc(jd, swe.MOON))
print(swe.calc(jd, swe.SATURN))
print(swe.calc(jd, swe.MERCURY))
print(swe.calc(jd, swe.VENUS))
print(swe.calc(jd, swe.TRUE_NODE))
print(swe.calc_ut(jd, swe.MARS))
print("==============")
print(swe.calc_ut(jd, swe.JUPITER, swe.FLG_SWIEPH))




63.65217221552204
Sun:238.55975708947915
Sun with UTC calculation : 238.5605683363985
Moon:178.32926926808636
Saturn: 331.17814001205124
jupiter : 89.19687179392292
mercury : 219.0886818280194
venus : 233.0035789284637
rahu : 318.7776269786655
ketu : 342.9972629239083
((113.4165077391657, 0.19464427072276078, 4.334262139339734, -0.10072714136383641, 0.002590237733368064, -0.00773940285692651), 260)
((262.7793930347219, -7.187739462519808e-05, 0.9843811812419605, 1.0172243492334392, -1.6063024637808265e-05, -9.553338833181857e-05), 260)
((202.54890521332914, -3.2056281399721884, 0.0026925895460254774, 11.9353098009088, -0.8068852155555623, 1.7924231182517502e-05), 260)
((355.397775957294, -2.319137956333152, 9.428226830475914, 0.029177858167942022, 0.0035855469350657134, 0.016564869148639673), 260)
((243.30831777326216, 1.5959949820534824, 1.1513640340393132, 1.3186994190489816, -0.12414273781908144, 0.018881590527301942), 260)
((257.22321487370647, 0.18270720848211522, 1.69905569845974

In [13]:
rashi_order = ['Mesh', 'Vrishabh', 'Mithun', 'Kark', 'Simha', 'Kanya', 
                'Tula', 'Vrishchik', 'Dhanu', 'Makar', 'Kumbh', 'Meen']
abs(rashi_order.index('Mesh')-rashi_order.index('Meen'))+1

12

In [2]:
import swisseph as swe
import pandas as pd
from datetime import datetime, timedelta

# Define the tithi names
TITHI_NAMES = [
    "Pratipat", "Dwitiya", "Tritiya", "Chaturthi", "Panchami",
    "Shashthi", "Saptami", "Ashtami", "Navami", "Dashami",
    "Ekadashi", "Dwadashi", "Trayodashi", "Chaturdashi", "Purnima",
    "Pratipat", "Dwitiya", "Tritiya", "Chaturthi", "Panchami",
    "Shashthi", "Saptami", "Ashtami", "Navami", "Dashami",
    "Ekadashi", "Dwadashi", "Trayodashi", "Chaturdashi", "Amavasya"
]

# Set the ayanamsa to Lahiri/Chitrapaksha
swe.set_sid_mode(swe.SIDM_LAHIRI)

def get_julian(year, month, day, hour, minute):
    return swe.julday(year, month, day, hour + minute/60.0)

def jd_to_datetime_correct(jd, tz_offset=5.5):
    """CORRECT conversion from Julian Day to datetime"""
    # Julian Day to Gregorian date conversion
    jd = jd + 0.5
    z = int(jd)
    f = jd - z
    
    if z < 2299161:
        a = z
    else:
        alpha = int((z - 1867216.25) / 36524.25)
        a = z + 1 + alpha - int(alpha / 4)
    
    b = a + 1524
    c = int((b - 122.1) / 365.25)
    d = int(365.25 * c)
    e = int((b - d) / 30.6001)
    
    day = b - d - int(30.6001 * e) + f
    hour = (day - int(day)) * 24
    day = int(day)
    
    if e < 14:
        month = e - 1
    else:
        month = e - 13
        
    if month > 2:
        year = c - 4716
    else:
        year = c - 4715
    
    # Create datetime object
    dt = datetime(year, month, day)
    
    # Add hours, minutes, seconds
    hours = int(hour)
    minutes = int((hour - hours) * 60)
    seconds = int(((hour - hours) * 60 - minutes) * 60)
    
    dt = dt.replace(hour=hours, minute=minutes, second=seconds)
    
    # Apply timezone offset
    dt = dt + timedelta(hours=tz_offset)
    
    return dt

def get_tithi_info(diff_value):
    """Determine paksha and tithi based on the difference value between sun and moon"""
    tithi_number = int((diff_value % 30) + 1)
    
    if tithi_number <= 15:
        paksha = "Shukla Paksha"
    else:
        paksha = "Krishna Paksha"
    
    tithi_name = TITHI_NAMES[tithi_number - 1]
    
    return tithi_number, tithi_name, paksha

def find_tithi_end_time(start_jd, current_diff):
    """Find the exact time when the current tithi ends"""
    # Calculate the target difference for the next tithi
    next_tithi_target = (int(current_diff) + 1) * 12
    
    if next_tithi_target >= 360:
        next_tithi_target = 360
    
    # Use binary search to find the exact time
    low_jd = start_jd
    high_jd = start_jd + 2.0
    tolerance = 1e-6
    
    for _ in range(50):
        mid_jd = (low_jd + high_jd) / 2.0
        
        # Calculate positions at midpoint with Lahiri ayanamsa
        ayanamsa = swe.get_ayanamsa_ut(mid_jd)
        sun_pos, _ = swe.calc_ut(mid_jd, swe.SUN)
        moon_pos, _ = swe.calc_ut(mid_jd, swe.MOON)
        
        sun_lon = (sun_pos[0] - ayanamsa) % 360
        moon_lon = (moon_pos[0] - ayanamsa) % 360
        
        if sun_lon < moon_lon:
            current_diff_mid = (moon_lon - sun_lon)
        else:
            current_diff_mid = (360 - sun_lon + moon_lon)
        
        if abs(current_diff_mid - next_tithi_target) < tolerance:
            break
            
        if current_diff_mid < next_tithi_target:
            low_jd = mid_jd
        else:
            high_jd = mid_jd
    
    return mid_jd

def hours_to_hh_mm(hours):
    """Convert decimal hours to hours:minutes format"""
    total_minutes = int(hours * 60)
    h = total_minutes // 60
    m = total_minutes % 60
    return f"{h:02d}:{m:02d}"

# Set timezone offset for IST
tz = 5.5

# Input time is already in IST (Local time)
current_time_ist = pd.to_datetime("2025-09-17 22:51:00")
print(f"Current Time (IST): {current_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")

# Convert IST to UTC for calculations
current_time_utc = current_time_ist - timedelta(hours=tz)
print(f"Current Time (UTC): {current_time_utc.strftime('%Y-%m-%d %H:%M:%S')}")

# Calculate Julian Day in UTC
jd = get_julian(current_time_utc.year, current_time_utc.month, current_time_utc.day, 
                current_time_utc.hour, current_time_utc.minute)

# Calculate ayanamsa and planetary positions
ayanamsa = swe.get_ayanamsa_ut(jd)
print(f"Ayanamsa (Lahiri): {ayanamsa:.6f}°")

sun_pos, _ = swe.calc_ut(jd, swe.SUN)
moon_pos, _ = swe.calc_ut(jd, swe.MOON)

# Adjust longitudes for ayanamsa
sun_lon = (sun_pos[0] - ayanamsa) % 360
moon_lon = (moon_pos[0] - ayanamsa) % 360

# Calculate the difference between moon and sun longitude
if sun_lon < moon_lon:
    diff = (moon_lon - sun_lon)
else:
    diff = (360 - sun_lon + moon_lon)

diff_normalized = diff / 12
print(f"Sun longitude: {sun_lon:.6f}°")
print(f"Moon longitude: {moon_lon:.6f}°")
print(f"Absolute difference: {diff:.6f}°")
print(f"Difference value (tithi units): {diff_normalized:.6f}")

# Get tithi information
tithi_number, tithi_name, paksha = get_tithi_info(diff_normalized)

print(f"\nTithi Information:")
print(f"Tithi Number: {tithi_number}")
print(f"Tithi Name: {tithi_name}")
print(f"Paksha: {paksha}")
print(f"Full Description: {tithi_name} ({paksha})")

# Find when this tithi ends (in UTC)
tithi_end_jd_utc = find_tithi_end_time(jd, diff_normalized)

# Convert end time back to IST using CORRECT conversion
tithi_end_time_ist = jd_to_datetime_correct(tithi_end_jd_utc, tz)

print(f"\nTITHI END TIME (IST):")
print(f"Tithi End Time (IST): {tithi_end_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")

# Calculate time remaining
time_remaining = tithi_end_time_ist - current_time_ist
total_seconds = time_remaining.total_seconds()
hours_remaining = total_seconds / 3600
minutes_remaining = total_seconds / 60

# Convert to HH:MM format
h = int(hours_remaining)
m = int((hours_remaining - h) * 60)
time_remaining_hh_mm = f"{h:02d}:{m:02d}"

print(f"\nTIME REMAINING CALCULATION:")
print(f"Current Time (IST): {current_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tithi End Time (IST): {tithi_end_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Time remaining: {time_remaining_hh_mm} (HH:MM)")
print(f"Time remaining: {hours_remaining:.2f} hours")
print(f"Time remaining: {minutes_remaining:.0f} minutes")

# Compare with third-party app result
app_end_time_ist = pd.to_datetime("2025-09-17 23:38:51")  # Assuming this is IST
print(f"\nCOMPARISON WITH THIRD-PARTY APP:")
print(f"Our calculation end time (IST): {tithi_end_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Third-party app end time (IST): {app_end_time_ist.strftime('%Y-%m-%d %H:%M:%S')}")

time_difference = tithi_end_time_ist - app_end_time_ist
print(f"Difference between calculations: {time_difference.total_seconds()/60:.1f} minutes")


# Additional information about special tithis
if tithi_number == 15:
    print("\nNote: Today is Purnima (Full Moon) - considered highly auspicious.")
elif tithi_number == 30:
    print("\nNote: Today is Amavasya (New Moon) - significant for ancestral rituals.")
elif tithi_number == 11 or tithi_number == 26:
    print("\nNote: Today is Ekadashi - auspicious for fasting and spiritual practices.")

Current Time (IST): 2025-09-17 22:51:00
Current Time (UTC): 2025-09-17 17:21:00
Ayanamsa (Lahiri): 24.216277°
Sun longitude: 150.857125°
Moon longitude: 102.439382°
Absolute difference: 311.582257°
Difference value (tithi units): 25.965188

Tithi Information:
Tithi Number: 26
Tithi Name: Ekadashi
Paksha: Krishna Paksha
Full Description: Ekadashi (Krishna Paksha)

TITHI END TIME (IST):
Tithi End Time (IST): 2025-09-17 23:40:06

TIME REMAINING CALCULATION:
Current Time (IST): 2025-09-17 22:51:00
Tithi End Time (IST): 2025-09-17 23:40:06
Time remaining: 00:49 (HH:MM)
Time remaining: 0.82 hours
Time remaining: 49 minutes

COMPARISON WITH THIRD-PARTY APP:
Our calculation end time (IST): 2025-09-17 23:40:06
Third-party app end time (IST): 2025-09-17 23:38:51
Difference between calculations: 1.2 minutes

Note: Today is Ekadashi - auspicious for fasting and spiritual practices.


In [68]:
import pytz

today = datetime.now(pytz.timezone("Asia/Kolkata"))
print(today)
weekday_index = (today.weekday() + 1) % 7  # Sunday=0 … Saturday=6
print(weekday_index)
WEEKDAYS = ["Ravivara", "Somavara", "Mangalavara", "Budhavara", "Guruvara", "Shukravara", "Shanivara"]

weekday = WEEKDAYS[weekday_index]
print(weekday)

2025-09-17 23:55:03.705071+05:30
3
Budhavara


In [1]:
from timezonefinder import TimezoneFinder
from geopy.geocoders import Nominatim

def test_geocoding(city_name="Jaunpur, Uttar Pradesh"):
    """Test geocoding for a city name"""
    print(f"Testing geocoding for: '{city_name}'")
    print("-" * 50)
    
    try:
        # Initialize geocoder
        geolocator = Nominatim(user_agent="test_geocoding_app")
        
        # Geocode the location
        print("Geocoding...")
        location = geolocator.geocode(city_name)
        
        if location:
            print("✅ Success! Found location:")
            print(f"  Address: {location.address}")
            print(f"  Latitude: {location.latitude}")
            print(f"  Longitude: {location.longitude}")
            
            # Get timezone
            tf = TimezoneFinder()
            timezone_name = tf.timezone_at(lat=location.latitude, lng=location.longitude)
            print(f"  Timezone: {timezone_name}")
            
            # Additional location details
            print(f"  Raw data: {location.raw}")
            
        else:
            print("❌ No location found!")
            
    except Exception as e:
        print(f"❌ Error: {e}")

test_geocoding("Jaunpur, Uttar Pradesh")
    
print("\n" + "="*50)
print("Testing with other variations:")
print("="*50)


ModuleNotFoundError: No module named 'timezonefinder'